# Rare Cell Type Analysis — Part 01: Subset and Annotate (CPU)

This notebook pulls one brain region out of the ASAP CRN PMDBS sc/snRNA-seq
cohort and annotates it against a reference taxonomy. The worked example uses
dopaminergic neurons in the Substantia Nigra; change the configuration cell to
target another region or population.

The workflow:

1. Selects samples for the target region from CRN metadata.
2. Uses CRN `.final.h5ad` files as the curated, QC-filtered cell set.
3. Recovers full-gene raw counts for those cells from the matching
   `.merged_cleaned_unfiltered.h5ad` files.
4. Audits cohort composition (datasets, samples, conditions) and saves a checkpoint.
5. Normalizes counts and selects features chosen to keep the target population
   resolvable after integration.
6. Annotates cells with MapMyCells using a region-appropriate reference taxonomy
   (basal ganglia for the worked example).
7. Validates target-population labels against orthogonal marker genes.

> **Run in:** CPU app (high memory).
> **Output:** annotated AnnData saved to the workspace bucket, used by Part 02.

CRN pipeline documentation:
- Repository: https://github.com/ASAP-CRN/sc-rnaseq-wf
- Guide: https://asap-crn.github.io/asap-crn-learning-lab/pipeline/

Provenance:
- Validated against CRN release v5.0.0 on 2026-09-21. The notebook selects the
  latest available release automatically, so newer releases may give slightly
  different results.

##  Imports, analysis config, and reproducibility

In [ ]:
# -- Imports -----------
from __future__ import annotations

import os
import re
import gc
import json
import random
from pathlib import Path
from urllib.parse import urlparse
from urllib.request import urlretrieve
from packaging.version import Version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import anndata as ad
import scanpy as sc
from google.cloud import storage

# -- Reproducibility ----------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)   # use for explicit NumPy sampling

# -- Provenance ----------
ANALYSIS_RUN_DATE = "2026-10-01"
METADATA_RELEASE  = "v5.0.0"   # metadata release this notebook was validated against.
                               # Expression-file releases are resolved at runtime (below)
                               # and recorded so reruns remain auditable.

# -- Display / Scanpy settings ----------
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
sc.settings.verbosity = 1

# Optional: per-cell timing
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install -q "ipython-autotime>=0.3.2"
    %load_ext autotime


## Analysis configuration

Everything specific to the worked example (Substantia Nigra, dopaminergic neurons) is
set in the next cell. **To target another region or rare population, edit only this cell.**
The rest of the notebook reads from these variables.

In [ ]:
# -- Target region ----------
# A sample is selected if ANY region-name column contains TARGET_REGION_SUBSTR
# (case-insensitive) OR any ontology column matches TARGET_ONTOLOGY_IDS.
ANALYSIS_NAME        = "sn_dopaminergic"      # used for folder, file, and bucket names
TARGET_REGION_SUBSTR = "substantia nigra"
TARGET_ONTOLOGY_IDS  = {
    "UBERON:0002038",   # substantia nigra
    "UBERON:0001965",   # substantia nigra pars compacta
}
# Example, striatum:   TARGET_REGION_SUBSTR = "putamen";  TARGET_ONTOLOGY_IDS = {"UBERON:0001874"}

# -- Target population ----------
TARGET_LABEL = "Dopaminergic"       # simplified MapMyCells cell_type of the rare population
MARKER_GENES = ["TH", "SLC6A3", "SLC18A2", "DDC", "NR4A2", "KCNJ6", "SNCA", "ALDH1A1", "GCH1"]

# -- Feature selection / batch ----------
BATCH_KEY   = "ASAP_dataset_id"     # also the scVI batch key in Part 02
N_TOP_GENES = 3000

# -- Reference taxonomy for MapMyCells ----------
# Allen Institute HMBA Basal Ganglia taxonomy: fits SN and striatum.
# For a cortical region, point these at a cortical taxonomy and update class_mapper.
REFERENCE = {
    "name": "HMBA-BG",
    "precomputed_stats_url": (
        "https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com/"
        "HMBA/BasalGanglia/BICAN_05072025_pre-print_release/MapMyCells/Human.precomputed_stats.20250507.h5"),
    "query_markers_url": (
        "https://released-taxonomies-802451596237-us-west-2.s3.us-west-2.amazonaws.com/"
        "HMBA/BasalGanglia/BICAN_05072025_pre-print_release/MapMyCells/Human.query_markers.20250507.json"),
    # reference Class_name -> simplified cell_type
    "class_mapper": {
        "OPC-Oligo": "OPC-Oligo",   "Astro-Epen": "Astrocyte",   "Vascular": "Vascular",
        "Immune": "Immune",         "F M Glut": "Glutamatergic", "M Dopa": "Dopaminergic",
        "CN CGE GABA": "GABAergic", "F M GABA": "GABAergic",     "CN MGE GABA": "GABAergic",
        "CN LGE GABA": "GABAergic", "Cx GABA": "GABAergic",      "CN GABA-Glut": "GABAergic",
    },
    "min_rho":  0.2,   # correlation below this -> cell_type "Unknown"
    "min_prob": 0.5,   # bootstrap probability below this -> "Unknown"
}
assert TARGET_LABEL in set(REFERENCE["class_mapper"].values()), \
    f"TARGET_LABEL {TARGET_LABEL!r} is not produced by REFERENCE['class_mapper']"


## Set Paths

In [ ]:
# Paths (local disk of THIS app)
HOME    = Path.home() / "workspace"
WS_ROOT = HOME / "ws_files" / ANALYSIS_NAME

CRN_METADATA_DIR    = HOME / "release_resources" / "cohort-pmdbs-sc-rnaseq"
METADATA_TABLES_DIR = CRN_METADATA_DIR / "metadata" / "release"
FILEMETADATA_DIR    = CRN_METADATA_DIR / "file_metadata" / "release"

SAMPLE   = METADATA_TABLES_DIR / METADATA_RELEASE / "SAMPLE.csv"
SUBJECT  = METADATA_TABLES_DIR / METADATA_RELEASE / "SUBJECT.csv"
CLINPATH = METADATA_TABLES_DIR / METADATA_RELEASE / "CLINPATH.csv"

OUTPUT_DIR   = WS_ROOT / "output";     OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
download_dir = WS_ROOT / "downloads";  download_dir.mkdir(parents=True, exist_ok=True)   # scratch
MMC_DIR      = WS_ROOT / "mapmycells"; MMC_DIR.mkdir(parents=True, exist_ok=True)

print("CRN metadata:  ", CRN_METADATA_DIR)
print("Local outputs: ", OUTPUT_DIR)


## Load & harmonize CRN metadata

The CRN release ships various metadata tables, we utilize 3 — **SAMPLE** (one row per sequenced
sample), **SUBJECT** (donor demographics), and **CLINPATH** (clinical/pathology).
We load SAMPLE first and clean its region annotations, then select the target-region samples and join the other two.

In [ ]:
# Load PMDBS Cohort Sample Data 
sample = pd.read_csv(SAMPLE , low_memory=False)
print(f"{SAMPLE.name}: {sample.shape[0]:,} rows x {sample.shape[1]} columns")
sample.head(3)

### 2.1 Harmonize Simple Brain Region Labels 
CRN region fields arrive as `"Name (ABBREV, UBERON:…)"`. We split them into
name / abbrev / ontology-id.

In [ ]:
# Region fields arrive as "Name (ABBREV, UBERON:0000000)".
# Split them naively apart so we can filter on the plain name.
for i in [1, 2, 3]:
    col = f"region_level_{i}"
    sample[[f"{col}_name", f"{col}_abbrev", f"{col}_ontology_term_id"]] = (
        sample[col].str.extract(r"^(.*?) \((.*?), (.*?)\)$")
    )

# Report values that were populated but did NOT match the expected format. A
# silent NaN here would otherwise be indistinguishable from a missing region.
for i in [1, 2, 3]:
    col = f"region_level_{i}"
    failed = sample[sample[col].notna() & sample[f"{col}_name"].isna()]
    if len(failed):
        print(f"WARNING: {len(failed)} {col} value(s) did not match "
              f'"Name (ABBREV, UBERON:...)" and were left unparsed:')
        print("  ", sorted(failed[col].unique())[:10])
    else:
        print(f"{col}: all populated values parsed cleanly.")

print("\nregion_level_1_name:\n", sample["region_level_1_name"].value_counts(dropna=False))
print("\nregion_level_2_name:\n", sample["region_level_2_name"].value_counts(dropna=False))
print("\nregion_level_3_name:\n", sample["region_level_3_name"].value_counts(dropna=False))


In [ ]:
sample.groupby(["ASAP_dataset_id", "region_level_2"]).size().sort_values(ascending=False)

In [ ]:
# Level-2 mapping: the primary region label 
brain_simple_L2 = {
    "Prefrontal cortex":        "frontal_ctx",
    "Middle frontal gyrus":     "frontal_ctx",
    "Anterior cingulate gyrus": "cingulate_ctx",
    "Inferior parietal lobule": "parietal_ctx",
    "Middle temporal gyrus":    "temporal_ctx",
    "Hippocampus":              "subcortical",
    "Amygdala":                 "subcortical",
    "Putamen":                  "subcortical",
    "Substantia nigra":         "subcortical",
}
sample["brain_region_simple"] = sample["region_level_2_name"].map(brain_simple_L2)

# surface any L2 name unmapped
unknown_L2 = set(sample["region_level_2_name"].dropna()) - set(brain_simple_L2)
if unknown_L2:
    print("WARNING: unmapped region_level_2_name (add to brain_simple_L2):",
          sorted(unknown_L2))

# level-1 fallback: coarse lobe label ONLY where L2 is absent
brain_simple_L1 = {
    "Frontal lobe":            "frontal_ctx",
    "Cingulate gyrus":         "cingulate_ctx",
    "Temporal lobe":           "temporal_ctx",
    "Parietal lobe":           "parietal_ctx",
    "Striatum":                "subcortical",
    "Midbrain mesencephalon":  "subcortical",   # SN/VTA sit here
}
need = sample["region_level_2_name"].isna() & sample["region_level_1_name"].notna()
sample.loc[need, "brain_region_simple"] = sample.loc[need, "region_level_1_name"].map(brain_simple_L1)

# surface any fallback L1 lobe unmapped 
unknown_L1 = set(sample.loc[need, "region_level_1_name"].dropna()) - set(brain_simple_L1)
if unknown_L1:
    print("WARNING: unmapped region_level_1_name used for fallback "
          "(add to brain_simple_L1):", sorted(unknown_L1))

# provenance of each label: precise L2, coarse L1 fallback, or none.
sample["region_label_source"] = np.select(
    [sample["region_level_2_name"].notna() & sample["brain_region_simple"].notna(),
     sample["brain_region_simple"].notna()],
    ["L2", "L1_fallback"], default="none")

print(sample["region_label_source"].value_counts())
print("still unlabeled:", int(sample["brain_region_simple"].isna().sum()))

In [ ]:
# Multi-compartment level-3 annotation (GM + WM in one sample) is valid; give it
# a canonical combined label rather than dropping it. Note the separator is ';'
# with no trailing space.
COMBINED_REGION = {
    "Grey matter (GM, UBERON:0002020);White matter (WM, UBERON:0002316)":
        ("Grey and white matter", "GM+WM", "UBERON:0002020;UBERON:0002316"),
}
for raw, (nm, ab, on) in COMBINED_REGION.items():
    hit = sample["region_level_3"].eq(raw)
    sample.loc[hit, "region_level_3_name"] = nm
    sample.loc[hit, "region_level_3_abbrev"] = ab
    sample.loc[hit, "region_level_3_ontology_term_id"] = on

In [ ]:
sample = sample.convert_dtypes().dropna(axis=1, how="all")
sample.head(5)

### Define the target-region cohort


In [ ]:
def _norm(s: pd.Series) -> pd.Series:
    """Lowercase + strip a column for case-insensitive matching."""
    return s.astype("string").str.strip().str.lower()

NAME_COLS     = [c for c in sample.columns if c.endswith("_name")] + \
                [c for c in ("brain_region_simple", "tissue") if c in sample.columns]
ONTOLOGY_COLS = [c for c in sample.columns if c.endswith("ontology_term_id")]

target_ids  = {x.lower() for x in TARGET_ONTOLOGY_IDS}
region_mask = pd.Series(False, index=sample.index)
for c in NAME_COLS:     region_mask |= _norm(sample[c]).str.contains(TARGET_REGION_SUBSTR, na=False)
for c in ONTOLOGY_COLS: region_mask |= _norm(sample[c]).isin(target_ids)

region_samples = sample.loc[region_mask].copy()
print(f"{TARGET_REGION_SUBSTR!r} samples: {region_mask.sum():,} of {len(sample):,} rows")
assert len(region_samples) > 0, "no samples matched -- check TARGET_REGION_SUBSTR / TARGET_ONTOLOGY_IDS"


In [ ]:
region_samples.ASAP_team_id.value_counts()

### Define Function for Merging Metadata Tables

In [ ]:
## Functions for merging tables

MISSING_LABELS = {
    "",
    "NA",
    "N/A",
    "NAN",
    "NONE",
    "NULL",
    "NOT REPORTED",
    "UNKNOWN",
}


def standardize_missing(data: pd.DataFrame) -> pd.DataFrame:
    """Convert blank strings and common missing labels to pd.NA."""
    result = data.copy()
    text_columns = result.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        cleaned = result[column].astype("string").str.strip()
        missing_mask = cleaned.str.upper().isin(MISSING_LABELS)
        result[column] = cleaned.mask(missing_mask, pd.NA)

    return result
    
def collapse_duplicate_records(
    data: pd.DataFrame,
    keys: Sequence[str],
    table_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Collapse duplicate keys while preserving agreed complementary values.

    Selection and fill rules:
    1. Select the row with the most populated non-key fields as the base.
    2. If a field has exactly one distinct non-null value in the group,
       place that value in the collapsed row.
    3. If a field has multiple distinct values, retain the selected base
       value and record all observed values in the conflict report.
    4. Record ties when multiple rows are equally complete.
    """
    work = standardize_missing(data)
    metadata_columns = [
        column for column in work.columns
        if column not in keys
    ]

    work["_fields_present"] = (
        work[metadata_columns]
        .notna()
        .sum(axis=1)
    )
    work["_source_row"] = np.arange(len(work))

    selected_records = []
    conflict_records = []
    tie_records = []

    group_key = keys[0] if len(keys) == 1 else list(keys)

    for key_values, group in work.groupby(
        group_key,
        dropna=False,
        sort=False,
    ):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)

        key_record = dict(zip(keys, key_values))
        max_fields = int(group["_fields_present"].max())

        top_rows = (
            group.loc[
                group["_fields_present"].eq(max_fields)
            ]
            .sort_values("_source_row")
        )
        selected = top_rows.iloc[0].copy()

        if len(top_rows) > 1:
            tie_records.append({
                **key_record,
                "table": table_name,
                "n_tied_rows": len(top_rows),
                "fields_present": max_fields,
                "source_rows": " | ".join(
                    top_rows["_source_row"]
                    .astype(str)
                    .tolist()
                ),
            })

        conflicting_fields = []

        for column in metadata_columns:
            values = (
                group[column]
                .dropna()
                .drop_duplicates()
            )

            if len(values) == 1:
                selected[column] = values.iloc[0]

            elif len(values) > 1:
                conflicting_fields.append(column)

                selected_top_values = (
                    top_rows[column]
                    .dropna()
                )
                if len(selected_top_values):
                    selected[column] = (
                        selected_top_values.iloc[0]
                    )
                elif pd.isna(selected[column]):
                    selected[column] = values.iloc[0]

                conflict_records.append({
                    **key_record,
                    "table": table_name,
                    "field": column,
                    "n_unique_values": len(values),
                    "observed_values": " | ".join(
                        repr(value)
                        for value in values.tolist()
                    ),
                    "selected_value": repr(
                        selected[column]
                    ),
                    "n_source_records": len(group),
                })

        selected["_source_record_count"] = len(group)
        selected["_selection_tie"] = len(top_rows) > 1
        selected["_n_conflicting_fields"] = len(
            conflicting_fields
        )
        selected["_conflicting_fields"] = (
            " | ".join(conflicting_fields)
            if conflicting_fields
            else pd.NA
        )

        selected_records.append(selected)

    collapsed = (
        pd.DataFrame(selected_records)
        .drop(
            columns=[
                "_fields_present",
                "_source_row",
            ]
        )
        .reset_index(drop=True)
    )
    conflicts = pd.DataFrame(conflict_records)
    ties = pd.DataFrame(tie_records)

    if collapsed.duplicated(list(keys)).any():
        raise AssertionError(
            f"{table_name} cleaning did not produce "
            "one row per key."
        )

    return collapsed, conflicts, ties


### Merge region sample data with SUBJECT metadata


In [ ]:
# Load PMDBS Cohort Subject Metadata Table 

subject = pd.read_csv(SUBJECT , low_memory=False)
print(f"{SUBJECT.name}: {subject.shape[0]:,} rows x {subject.shape[1]} columns")
subject.head(3)

In [ ]:
expected_sample_rows = len(region_samples)

subject_columns = [
    "ASAP_dataset_id",
    "ASAP_subject_id",
    "subject_id",
    "source_subject_id",
    "sex",
    "race",
    "ethnicity",
    "biobank_name",
    "gp2_phenotype",
]


subject_keys = [
    "ASAP_subject_id",
    "ASAP_dataset_id",
    "subject_id",
]


subject_clean, subject_conflicts, subject_selection_ties = collapse_duplicate_records(
    subject[subject_columns],
    subject_keys,
    table_name="SUBJECT"
)

print("Original SUBJECT rows:", len(subject))
print("Clean SUBJECT rows:", len(subject_clean))
print("SUBJECT conflicts:", len(subject_conflicts))
print("SUBJECT top-record ties:", len(subject_selection_ties))

df = pd.merge(
    region_samples,
    subject_clean,
    on=subject_keys,
    how="left",
    validate="many_to_one",
    indicator="_subject_merge",
)

assert len(df) == expected_sample_rows, "SUBJECT merge changed the SAMPLE row count."

### Merge region sample data with CLINPATH metadata


In [ ]:
# Load PMDBS Cohort Sample Data 

clinpath = pd.read_csv(CLINPATH , low_memory=False)
print(f"{CLINPATH.name}: {clinpath.shape[0]:,} rows x {clinpath.shape[1]} columns")
clinpath.head(3)

In [ ]:
clin_key = ["ASAP_dataset_id", "ASAP_subject_id"]
clin_grp = clinpath.groupby(clin_key)

# Rows-per-subject: a uniform "4" points to a structural merge artifact, not
# per-record error. The odd count out is the one to inspect by hand.
print("Rows per within-dataset subject:")
print(clin_grp.size().value_counts().sort_index(), "\n")

# Is there a column that is systematically distinct across the duplicate rows?
# If yes -> that's a real dimension (visit/instrument) and collapsing loses it.
# If no  -> the rows are fragments differing only by which fields are filled.
clin_dupe = clin_grp.filter(lambda g: len(g) > 1)
clin_value_cols = [c for c in clinpath.columns if c not in clin_key + ["ASAP_team_id",
              "subject_id", "source_subject_id"]]
per_group_max_distinct = (clin_dupe.groupby(clin_key)[clin_value_cols].nunique(dropna=True).max()
                              .sort_values(ascending=False))
print("Max distinct values any duplicate group shows, per field (top 8):")
print(per_group_max_distinct.head(8))
print("\n-> No column distinct in *every* row = fragmentation, safe to collapse.")

# Surface the odd-count subject explicitly (don't let .iloc[0] skip it)
odd = clin_grp.filter(lambda g: len(g) == 2)
if len(odd):
    print("\nOdd-count (2-row) subject — inspect directly:")
    display(odd.T)

In [ ]:
# Apply the agreed vocabulary maps FIRST so synonym-only differences
# (e.g. 'No PD nor other neurological disorder' == 'Healthy Control') do not
# masquerade as conflicts. Fill VOCAB_MAPS from the value_counts scan.

VOCAB_MAPS = {
    # (Deliberately empty / no primary_diagnosis mapping applied — see the
    #  "candidate harmonizations" note below. Add entries only once confirmed.)
}

CLINPATH_h = clinpath.copy()
for col, mapping in VOCAB_MAPS.items():
    if col in CLINPATH_h.columns:
        CLINPATH_h[col] = CLINPATH_h[col].replace(mapping)

# Collapse to one row per PERSON. This absorbs BOTH within-dataset fragments and
# cross-dataset resubmissions in a single pass; complementary fields fill,
# genuine value-vs-value disagreements are recorded in CLINPATH_conflicts.
CLINPATH_clean, CLINPATH_conflicts, CLINPATH_ties = collapse_duplicate_records(
    CLINPATH_h, ["ASAP_subject_id"], table_name="CLINPATH")

clin_multi         = CLINPATH_clean[CLINPATH_clean["_source_record_count"] > 1]
complementary = clin_multi[clin_multi["_n_conflicting_fields"] == 0]
conflicted    = clin_multi[clin_multi["_n_conflicting_fields"] > 0]

print(f"Person-level rows:              {len(CLINPATH_clean)}")
print(f"Subjects collapsed from >1 row: {len(clin_multi)}")
print(f"  ├─ complementary (lossless):  {len(complementary)}   [Issue 1]")
print(f"  └─ TRUE disagreement:         {len(conflicted)}   [Issue 2 → review]")

In [ ]:
# One subject whose 4 rows each held different fields, collapsed to one complete row.
ex_id = complementary["ASAP_subject_id"].iloc[0]
raw = CLINPATH_h[CLINPATH_h["ASAP_subject_id"] == ex_id]

# show only the fields that were scattered (populated in some rows, null in others)
frag_cols = [c for c in clin_value_cols
             if raw[c].notna().any() and raw[c].isna().any()]

print(f"BEFORE — {ex_id}: {len(raw)} fragmented rows (note the scattered fills)")
display(raw[["ASAP_dataset_id"] + frag_cols].T)

print("AFTER — one reconstructed record (values filled from across the rows):")
display(CLINPATH_clean.loc[CLINPATH_clean["ASAP_subject_id"] == ex_id, frag_cols].T)

In [ ]:
before_merge = len(df)

# carry only the clinical payload; drop dataset/team/local-id + audit columns so
# person-level facts don't overwrite each sample's own dataset context
payload = [c for c in CLINPATH_clean.columns
           if c not in {"ASAP_dataset_id", "ASAP_team_id",
                        "subject_id", "source_subject_id"}
           and not c.startswith("_")]

df = df.merge(CLINPATH_clean[payload], on="ASAP_subject_id",
              how="left", validate="many_to_one", indicator="_clinpath_merge")
assert len(df) == before_merge, "CLINPATH merge changed the row count."

print("CLINPATH merge coverage:")
print(df["_clinpath_merge"].value_counts())
df = df.drop(columns="_clinpath_merge")

In [ ]:
# Uniform missing representation before saving/harmonizing. SAMPLE columns
# were merged raw, so SAMPLE-origin fields may still hold "NA"/"" while
# SUBJECT-/CLINPATH-origin fields are already pd.NA. Standardize once here.
df = standardize_missing(df)
print("Standardized missing values across", df.shape[1], "columns.")
df.head()

In [ ]:
region_subset = df.reset_index(drop=True)
SAMPLES_CSV = OUTPUT_DIR / f"{ANALYSIS_NAME}__samples.csv"
region_subset.to_csv(SAMPLES_CSV, index=False)
print("saved:", SAMPLES_CSV)


##  Load Gene-Expression Matrices (.final.h5ad)

From the curated-files manifest we keep the `.final.h5ad` per selected dataset,
resolve each to its **newest release** (so reruns are reproducible), then load only
the cells belonging to our samples. Sample IDs in the expression files are normalized
to a canonical form (`canonical_sample_id()`) before matching.

Each `.final.h5ad` contains 3000 highly variable genes and already has been QC'.  [Refer to the pipeline guide for QC filters.](https://asap-crn.github.io/asap-crn-learning-lab/pipeline/)

In [ ]:
CURATED_FILES = ( CRN_METADATA_DIR / FILEMETADATA_DIR / METADATA_RELEASE / "curated_files.csv" )
curated_files = pd.read_csv(CURATED_FILES , low_memory=False)
print(f"{CURATED_FILES.name}: {curated_files.shape[0]:,} rows x {curated_files.shape[1]} columns")

# keep curated files from the datasets that contain target-region samples
dataset_ids = region_subset.ASAP_dataset_id.unique().tolist()
mask = curated_files.ASAP_dataset_id.isin(dataset_ids)
filtered_curated = curated_files[mask]
filtered_curated.head()

In [ ]:
# Find all rows where the 'filename' column ends with '.final.h5ad'
mask = filtered_curated['filename'].str.endswith('.final.h5ad', na=False)

# Filter the DataFrame
region_geneexp = filtered_curated[mask]
region_geneexp

#### Resolve exact CRN expression-file releases and URIs

In [ ]:
_client = storage.Client()

def gcs_get(uri: str, local_path) -> None:
    """Download gs://bucket/path to a local file."""
    parsed = urlparse(uri)                      # gs://bucket/path/to/file
    blob = _client.bucket(parsed.netloc).blob(parsed.path.lstrip("/"))
    blob.download_to_filename(str(local_path))

    
def gcs_ls(gcs_path: str) -> list[str]:
    """List immediate children (dirs + files) under a gs:// prefix."""
    parsed = urlparse(gcs_path if gcs_path.startswith("gs://") else f"gs://{gcs_path}")
    bucket, prefix = parsed.netloc, parsed.path.lstrip("/")
    if prefix and not prefix.endswith("/"):
        prefix += "/"
    it = _client.list_blobs(bucket, prefix=prefix, delimiter="/")
    blobs = list(it)  # must consume the iterator before .prefixes is populated
    return [f"gs://{bucket}/{p}" for p in it.prefixes] + \
           [f"gs://{bucket}/{b.name}" for b in blobs]


def gcs_put(local_path, uri: str) -> None:
    """Upload a local file to gs://bucket/path (hand-off to the GPU app)."""
    assert "<your-workspace-bucket>" not in uri, "set WORKSPACE_BUCKET in the paths cell"
    parsed = urlparse(uri)
    _client.bucket(parsed.netloc).blob(parsed.path.lstrip("/")).upload_from_filename(str(local_path))
    print("uploaded ->", uri)


In [ ]:
def latest_release_uri(gcp_uri, filename):
    base = re.sub(r"/cohort_analysis/?$", "", str(gcp_uri))
    versions = []
    for p in gcs_ls(f"{base}/release"):
        name = p.rstrip("/").rsplit("/", 1)[-1]
        try:
            Version(name.lstrip("v"))
            versions.append(name)
        except Exception:
            continue                      # skip non-version entries (e.g. 'latest/')
    if not versions:
        raise FileNotFoundError(f"no release versions under {base}/release")
    latest = max(versions, key=lambda v: Version(v.lstrip("v")))
    return f"{base}/release/{latest}/cohort_analysis/{filename}"

region_geneexp["full_uri"] = region_geneexp.apply(
    lambda r: latest_release_uri(r["gcp_uri"], r["filename"]), axis=1
)
gxp_gcp_uris = region_geneexp["full_uri"].tolist()
gxp_gcp_uris

In [ ]:
# sample ids wanted per dataset
groups = region_subset.groupby("ASAP_dataset_id")["ASAP_sample_id"].apply(
    lambda s: set(s.dropna().astype(str)))
print(groups.apply(len).rename("n_samples"))

# map each expression file to ITS OWN dataset's samples (no positional assumptions)
wanted_by_file = {}
for r in region_geneexp.itertuples():
    assert r.ASAP_dataset_id in groups.index, f"{r.ASAP_dataset_id}: file but no samples"
    wanted_by_file[r.full_uri] = groups[r.ASAP_dataset_id]

no_file = set(groups.index) - set(region_geneexp["ASAP_dataset_id"])
assert not no_file, f"datasets with samples but no .final.h5ad: {sorted(no_file)}"
print(f"{len(wanted_by_file)} expression files cover {len(groups)} datasets")


In [ ]:
def canonical_sample_id(obs_sample: pd.Series) -> pd.Series:
    """Canonical ASAP_PMDBS_XXXX_sYYY id: fix PMBDS typo, drop replicate suffix."""
    return (obs_sample.astype(str)
            .str.replace("PMBDS", "PMDBS", regex=False)
            .str.extract(r"(ASAP_PMDBS_\d+_s\d+)", expand=False))



adatas = []
for u in gxp_gcp_uris:
    local = download_dir / u.rsplit("/", 1)[-1]        # full path, not bare filename
    gcs_get(u, local)
    a = ad.read_h5ad(local, backed="r")

    a.obs["sample_id"] = canonical_sample_id(a.obs["sample"])
    want = set(map(str, wanted_by_file[u]))
    keep = a.obs["sample_id"].isin(want)
    a = a[keep.values].to_memory()
    a.obs["source_file"] = local.name                   # keep just the filename as the tag

    matched = set(a.obs["sample_id"].dropna())
    print(f"{local.name}: kept {a.n_obs:,} cells | "
          f"{len(matched & want)}/{len(want)} samples | "
          f"missing: {sorted(want - matched)[:5]}")
    adatas.append(a)


### Merge AnnData Objects

We concatenate the per-file objects, then attach the harmonized metadata **once**
on the merged object (joined by canonical sample id, so cells can't be mismatched).
A coverage check flags any cells that didn't receive metadata.

In [ ]:
#  Joined metadata lookup, keyed by canonical sample id 
assert df["ASAP_sample_id"].is_unique, "ASAP_sample_id is not unique"

metadata_cols = [
    "ASAP_dataset_id", "ASAP_subject_id", "condition_id", "gp2_phenotype",
    "primary_diagnosis", "sex",
    "region_level_1_name", "region_level_2_name", "region_level_3_name",
    "brain_region_simple",
]
metadata_cols = [c for c in metadata_cols if c in df.columns]
sample_metadata = df.set_index("ASAP_sample_id")[metadata_cols]

In [ ]:
#  Concatenate, then attach metadata ONCE on the merged object
merged = ad.concat(
    adatas, join="outer", label="source",
    keys=[a.obs["source_file"][0] for a in adatas], index_unique="-",
)

merged.obs["sample_id"] = merged.obs["sample_id"].astype("string")

# every cell must have a parseable sample_id before the join, or .map produces
# NaN rows that .apply(pd.Series) can misshape/misalign
n_bad = merged.obs["sample_id"].isna().sum()
if n_bad:
    print(f"⚠ {n_bad:,} cells have no parseable sample_id — "
          "check canonical_sample_id() against their obs['sample'] values")

# join via reindex (robust, no dict->Series reshape)
meta_join = sample_metadata.reindex(merged.obs["sample_id"].values)
meta_join.index = merged.obs_names                      # realign to cells
for col in metadata_cols:
    merged.obs[col] = meta_join[col].values
merged.obs["case_id"] = merged.obs["gp2_phenotype"]

# categoricals for clean plotting / scVI batch handling
for col in [c for c in ["condition_id", "sex", "brain_region_simple",
                        "primary_diagnosis", "ASAP_dataset_id"] if c in merged.obs]:
    merged.obs[col] = merged.obs[col].astype("category")

frac_missing = merged.obs["condition_id"].isna().mean()
print(f"\nmerged: {merged.n_obs:,} cells x {merged.n_vars:,} genes")
print(f"cells missing metadata: {merged.obs['condition_id'].isna().sum():,} ({frac_missing:.1%})")
if frac_missing > 0.01:
    print("⚠ >1% of cells got no metadata — check sample_id normalization per dataset")
print(merged.obs["source"].value_counts())

### QC overview

Standard per-cell QC distributions, split by source file so batch differences are
visible. These files are already pipeline-QC'd, so this is a sanity check rather than
a filtering step. We recommend inspecting before trusting downstream integration.

In [ ]:


qc_cols = ["n_genes_by_counts", "total_counts", "pct_counts_mt",
           "pct_counts_rb", "doublet_score"]
merged.obs[qc_cols].describe()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, ["n_genes_by_counts", "total_counts", "pct_counts_mt", "doublet_score"]):
    sc.pl.violin(merged, col, groupby="source", jitter=False, ax=ax, show=False, rotation=90)
plt.tight_layout(); plt.show()

In [ ]:
merged

## Recover full-gene counts

The `.final.h5ad` files are reduced to ~3,000 highly variable genes. For a *rare*
population this is a trap: HVG selection is driven by variance across *all* cells, so
markers of a small population (for DA neurons: TH, SLC6A3, SLC18A2, ...) get competed
out and are simply absent.

The cell below confirms the problem for `MARKER_GENES`, then recovers **all genes** from
the `.merged_cleaned_unfiltered.h5ad` files, aligned to the cells we already selected.


In [ ]:
present = [g for g in MARKER_GENES if g in merged.var_names]
print("markers in .final.h5ad var_names:", present)
print("missing:", sorted(set(MARKER_GENES) - set(present)))


In [ ]:
#  extract full-gene counts for the region cells, one file at a time
parts = []
for u in gxp_gcp_uris:
    key = u.rsplit("/", 1)[-1]
    full_uri = u.replace(".final.h5ad", ".merged_cleaned_unfiltered.h5ad")
    local = download_dir / full_uri.rsplit("/", 1)[-1]
    gcs_get(full_uri, local)

    full = ad.read_h5ad(local)                 # whole file in RAM (fast subsetting)

    # curated cells for this file, suffix stripped to match the full file's barcodes
    cur_names = merged.obs_names[merged.obs["source_file"] == key]
    bare = cur_names.str.replace(f"-{key}$", "", regex=True)
    bare = bare[bare.isin(full.obs_names)]     # keep only cells present in the full file

    sub = full[bare].copy()                    # full genes, region cells
    sub.obs_names = bare + f"-{key}"           # restore the suffix so it matches `merged`
    sub.X = sub.X.astype("float32")

    out = f"part_{key}"
    sub.write_h5ad(out)
    parts.append(out)

    del full, sub
    os.remove(local)
    gc.collect()

#  concatenate the full-gene parts, RETAINING var annotations
adata_full = ad.concat(
    [ad.read_h5ad(p) for p in parts],
    join="inner",
    merge="first",          # keep var columns (gene_id, feature_type, mt, rb, …)
)
adata_full.layers["counts"] = adata_full.X.copy()

# every full-gene cell must exist in merged before we copy its metadata over
missing = ~adata_full.obs_names.isin(merged.obs_names)
assert not missing.any(), f"{missing.sum()} full-gene cells absent from merged.obs"
adata_full.obs = merged.obs.loc[adata_full.obs_names].copy()

print(f"{adata_full.n_obs:,} cells x {adata_full.n_vars:,} genes")
print("markers present:", [g for g in MARKER_GENES if g in adata_full.var_names])
print("var columns:", list(adata_full.var.columns))       # confirm gene_id survived

#### Verify the recovered object

Full-gene count present, markers recovered, cells aligned to metadata, and counts are
raw integers. These checks catch the common failure modes (wrong matrix, dropped var
annotations, misaligned barcodes) before you build on the object.

In [ ]:
adata_full

In [ ]:
print(f"{adata_full.n_obs:,} cells x {adata_full.n_vars:,} genes")
print("DA markers present:", [g for g in MARKER_GENES if g in adata_full.var_names])

In [ ]:
print("adata_full cells:", adata_full.n_obs)
print("merged cells:  ", merged.n_obs)
print("overlap:", merged.obs_names.isin(adata_full.obs_names).mean())

In [ ]:
# obs is indexed by the same names as the object
assert (adata_full.obs_names == adata_full.obs.index).all()
# team composition should match what merged had (minus any dropped cells)
print(pd.crosstab(adata_full.obs["source_file"], adata_full.obs.get("team", "NA")))

In [ ]:
x = adata_full.layers["counts"]
sample = x[:100].toarray() if hasattr(x, "toarray") else x[:100]
print("max:", sample.max(), "| all integers:", np.allclose(sample, np.round(sample)))

In [ ]:
bc = adata_full.obs_names[0]
cols = [c for c in ["source_file","team","cell_type","sample"] if c in merged.obs]
print("adata_full:", adata_full.obs.loc[bc, cols].to_dict())
print("merged :", merged.obs.loc[bc, cols].to_dict())

#### Checkpoint: full-gene counts recovered

The download loop above is the slow, memory-heavy step. Save here so the MapMyCells
and marker sections can be re-run (or a crashed kernel resumed) without re-downloading.

In [ ]:
CKPT = OUTPUT_DIR / f"{ANALYSIS_NAME}__01_fullgene_counts.h5ad"
adata_full.write_h5ad(CKPT)
print("checkpoint:", CKPT)

# ---- To resume from here in a fresh kernel, run the Imports/Config/Paths/GCS-helper
# ---- cells, then:   adata_full = ad.read_h5ad(CKPT)


## Preprocessing and feature selection

Normalize once (log-normalized `.X`, raw counts kept in `layers["counts"]`), select HVGs
on the counts per `BATCH_KEY`, then **force `MARKER_GENES` into the HVG set** so they
survive integration regardless of their variance rank. `.X` is used for marker scoring
below; `counts` + the HVG flags feed scVI in Part 02.


In [ ]:
adata_full.X = adata_full.layers["counts"].copy()
sc.pp.normalize_total(adata_full, target_sum=1e4)
sc.pp.log1p(adata_full)

sc.pp.highly_variable_genes(adata_full, n_top_genes=N_TOP_GENES, batch_key=BATCH_KEY,
                            flavor="seurat_v3", layer="counts")

markers_in = [g for g in MARKER_GENES if g in adata_full.var_names]
adata_full.var.loc[markers_in, "highly_variable"] = True
print(f"forced {len(markers_in)} markers into HVGs:", markers_in)
print("HVGs total:", int(adata_full.var["highly_variable"].sum()))


## Annotate with MapMyCells

Reference-based annotation against the taxonomy chosen in `REFERENCE`. This runs on CPU
(`cell_type_mapper` does not use the GPU) and takes roughly 30 minutes on
~100k cells. MapMyCells needs **raw counts** and **Ensembl gene IDs** as `var_names`,
so we write a small query file rather than mapping `adata_full` directly.

Running annotation *before* scVI (Part 02) is deliberate: MapMyCells compares each
cell to the reference on its own, so its labels do not depend on the embedding, and
marker-validated labels become the seed labels for scANVI.

In [ ]:
#reference files (downloaded once, reused)
precomputed_stats_fp = MMC_DIR / Path(REFERENCE["precomputed_stats_url"]).name
query_markers_fp     = MMC_DIR / Path(REFERENCE["query_markers_url"]).name
for url, fp in [(REFERENCE["precomputed_stats_url"], precomputed_stats_fp),
                (REFERENCE["query_markers_url"],     query_markers_fp)]:
    if not fp.exists():
        print("downloading", fp.name); urlretrieve(url, fp)
print("reference:", REFERENCE["name"], "|", precomputed_stats_fp.name, query_markers_fp.name)

#query file: raw counts, Ensembl IDs as var_names
assert "gene_id" in adata_full.var, "need var['gene_id'] (Ensembl) for MapMyCells"
query = ad.AnnData(X=adata_full.layers["counts"].copy(),
                   obs=adata_full.obs[[]].copy(),
                   var=adata_full.var[["gene_id"]].copy())
query.var["gene_name"] = query.var_names
query.var_names = query.var["gene_id"].astype(str)
query.var_names_make_unique()

QUERY_PATH = MMC_DIR / f"{ANALYSIS_NAME}__mmc_query.h5ad"
query.write_h5ad(QUERY_PATH)
print("query:", QUERY_PATH, f"({query.n_obs:,} cells x {query.n_vars:,} genes)")
del query; gc.collect()


In [ ]:
# limit numerical-library threads so the mapper's own workers are not oversubscribed
for v in ("NUMEXPR_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS"):
    os.environ[v] = "5"

from cell_type_mapper.cli.from_specified_markers import FromSpecifiedMarkersRunner

MMC_JSON = MMC_DIR / f"{ANALYSIS_NAME}__mmc_mapping.json"
MMC_CSV  = MMC_DIR / f"{ANALYSIS_NAME}__mmc_mapping.csv"

mmc_config = {
    "query_path":           str(QUERY_PATH),
    "extended_result_path": str(MMC_JSON),
    "csv_result_path":      str(MMC_CSV),
    "verbose_csv":          True,
    "query_markers":        {"serialized_lookup": str(query_markers_fp)},
    "precomputed_stats":    {"path": str(precomputed_stats_fp)},
    "type_assignment": {
        "n_processors":        max(1, (os.cpu_count() or 4) // 2),
        "normalization":       "raw",
        "bootstrap_factor":    0.5,
        "bootstrap_iteration": 100,
    },
}
FromSpecifiedMarkersRunner(args=[], input_data=mmc_config).run()
print("MapMyCells done ->", MMC_CSV)


In [ ]:
def summarize_mmc_results(mmc: pd.DataFrame, ref: dict) -> pd.DataFrame:
    """Collapse HMBA-style MapMyCells output to a simplified cell_type + confidence.

    Expects the HMBA-BG column layout (Class_name / Subclass_name / Neighborhood_name).
    For another taxonomy, adjust the column names and ref["class_mapper"].
    """
    out = mmc.copy()
    out["phenotype"] = out["Class_name"].map(ref["class_mapper"])
    unmapped = sorted(set(out.loc[out["phenotype"].isna(), "Class_name"].dropna()))
    if unmapped:
        print("WARNING: reference classes not in class_mapper (left as NaN):", unmapped)

    out["rho"]  = out["Class_correlation_coefficient"]
    out["prob"] = out["Class_bootstrapping_probability"]
    # GABAergic classes are better resolved at the neighborhood level
    gaba = out["phenotype"] == "GABAergic"
    out.loc[gaba, "rho"]  = out.loc[gaba, "Neighborhood_correlation_coefficient"]
    out.loc[gaba, "prob"] = out.loc[gaba, "Neighborhood_bootstrapping_probability"]

    out["cell_type"] = out["phenotype"]
    out.loc[out["rho"]  < ref["min_rho"],  "cell_type"] = "Unknown"
    out.loc[out["prob"] < ref["min_prob"], "cell_type"] = "Unknown"

    out = out.rename(columns={"Neighborhood_name": "supertype_name",
                              "Class_name": "class_name",
                              "Subclass_name": "subclass_name"})
    return out[["cell_id", "cell_type", "phenotype", "rho", "prob",
                "class_name", "subclass_name", "supertype_name"]]

mmc_raw = pd.read_csv(MMC_CSV, comment="#")
mmc_res = summarize_mmc_results(mmc_raw, REFERENCE).set_index("cell_id")

# attach by cell id (never positionally)
missing = adata_full.obs_names.difference(mmc_res.index)
assert len(missing) == 0, f"{len(missing)} cells have no MapMyCells result"
mmc_res = mmc_res.loc[adata_full.obs_names]
for col in mmc_res.columns:
    adata_full.obs[f"mmc_{col}"] = mmc_res[col].values
adata_full.obs["cell_type"] = adata_full.obs["mmc_cell_type"].astype("category")

print(adata_full.obs["cell_type"].value_counts())
print(f"\nUnknown (low confidence): {(adata_full.obs['cell_type'] == 'Unknown').mean():.1%}")
print(f"{TARGET_LABEL}: {(adata_full.obs['cell_type'] == TARGET_LABEL).sum():,} cells")


In [ ]:
# composition of the target population across datasets and conditions
pd.crosstab(adata_full.obs[BATCH_KEY], adata_full.obs["cell_type"]).T


## Validate target-population labels with marker genes

MapMyCells labels and marker expression are *independent* evidence. Before using the
labels as scANVI seeds in Part 02, check that cells called `TARGET_LABEL` are the ones
expressing `MARKER_GENES`, and look for cells that express the markers but were not
called (these are the cells a rare-population analysis is most likely to lose).

In [ ]:
from sklearn.metrics import roc_auc_score

markers_in = [g for g in MARKER_GENES if g in adata_full.var_names]
sc.tl.score_genes(adata_full, markers_in, score_name="marker_score", use_raw=False)  # .X = log-norm
adata_full.obs["is_target"] = (adata_full.obs["cell_type"] == TARGET_LABEL).values

auc = roc_auc_score(adata_full.obs["is_target"], adata_full.obs["marker_score"])
print(f"ROC AUC, marker_score -> MapMyCells {TARGET_LABEL}: {auc:.3f}")
print(adata_full.obs.groupby("cell_type", observed=True)["marker_score"]
      .describe()[["count", "mean", "50%"]].sort_values("mean", ascending=False))

sc.pl.violin(adata_full, "marker_score", groupby="cell_type", rotation=90, show=True)
sc.pl.dotplot(adata_full, var_names=markers_in, groupby="cell_type",
              standard_scale="var", title=f"{TARGET_LABEL} markers by MapMyCells cell_type")


In [ ]:
# discordant cells: high marker score but not called TARGET_LABEL (candidates the
# reference missed), and called TARGET_LABEL with low score (possible false positives)
tgt = adata_full.obs.loc[adata_full.obs["is_target"], "marker_score"]
lo, hi = tgt.quantile(0.10), tgt.quantile(0.50)

missed = (~adata_full.obs["is_target"]) & (adata_full.obs["marker_score"] >= hi)
weak   =   adata_full.obs["is_target"]  & (adata_full.obs["marker_score"] <  lo)
print(f"not {TARGET_LABEL} but marker_score >= {TARGET_LABEL} median: {missed.sum():,} cells")
print(adata_full.obs.loc[missed, "cell_type"].value_counts().head())
print(f"\n{TARGET_LABEL} with marker_score below 10th pct: {weak.sum():,} cells")

adata_full.obs["marker_discordant"] = (missed | weak).values


In [ ]:
# seed labels for scANVI (Part 02): confident MapMyCells calls; everything else unlabeled
adata_full.obs["seed_label"] = adata_full.obs["cell_type"].astype(str)
adata_full.obs.loc[adata_full.obs["marker_discordant"], "seed_label"] = "Unknown"
print(adata_full.obs["seed_label"].value_counts())


## Save and hand off to Part 02 (GPU)

`adata_full` now carries: full-gene raw counts (`layers["counts"]`), log-normalized `.X`,
HVG flags with markers forced in, harmonized sample/subject/clinical metadata,
MapMyCells labels (`mmc_*`, `cell_type`), marker scores, and `seed_label` for scANVI.
It is saved to the scratch directory so `02_gpu_integrate_and_export.ipynb` can
load it from the GPU app.

In [ ]:
ANNOTATED = OUTPUT_DIR / f"{ANALYSIS_NAME}__01_annotated.h5ad"
adata_full.write_h5ad(ANNOTATED)

print(f"\n{adata_full.n_obs:,} cells x {adata_full.n_vars:,} genes")
print("obs columns for Part 02:", [c for c in ["ASAP_dataset_id", "condition_id", "cell_type",
                                                 "seed_label", "marker_score"] if c in adata_full.obs])
print("obsm/layers:", list(adata_full.obsm.keys()), list(adata_full.layers.keys()))